In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
from sqlalchemy import create_engine

In [ ]:
data = pd.read_csv("../Datasets/Processed/final_data.csv")

In [ ]:
data.shape

In [ ]:
data.sample(1)

## Creating Database

In [ ]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/olist"
)



In [ ]:
data.to_sql(
    name='final_data',
    con=engine,
    if_exists='replace',
    index=False
)

print("Upload successful")

In [ ]:
data.sample(5)

## 1. Univariate Analysis

In [ ]:
def stats(df, col):
  if pd.api.types.is_numeric_dtype(df[col]):

    print(col, "- Numerical Column")
    print()

    null_count = df[col].isnull().sum().item()
    print("\nNull values are: ",null_count)

    null_percent = round((null_count/df.shape[0])*100,2)
    print("\nNull values percentage is: ",null_percent)

    mean = df[col].mean()
    print("\nMean is: ",mean)

    median = df[col].median()
    print("\nMedian is: ",median)

    mode = df[col].mode()[0]
    print("\nMode is: ",mode)

    std = df[col].std()
    print("\nStandard Deviation is: ",std)

    var = df[col].var()
    print("\nVariance is: ",var)

    skew = df[col].skew()
    print("\nSkewness is: ",skew)

    kurt = df[col].kurt()
    print("\nKurtosis is: ",kurt)

  else:
    print(col,"- Categorical Column")
    print()
    null_count = df[col].isnull().sum().item()
    print("\nNull values are: ",df[col].isnull().sum().item())

    null_percent = round((null_count/df.shape[0])*100,2)
    print("\nNull values percentage is: ",null_percent)

    mode = df[col].mode()[0]
    print("\nMode is: ",mode)

    print("\nValue Counts are: \n")
    print(df[col].value_counts())

### Revenue 

In [ ]:
#### Total Revenue

In [ ]:
query = """
            SELECT SUM(total_price) AS Total_revenue
            FROM final_data
         """
total_revenue = pd.read_sql(query, engine)
print("Total revenue is : ",total_revenue.iloc[0,0])

In [ ]:
stats(data,"total_price")

In [ ]:
plt.figure(figsize= (6,4))
sns.set_palette("Set2")
sns.distplot(data["total_price"])
plt.title("Revenue Distibution", color = "red")
plt.xlabel("Prices")
plt.show()



In [ ]:
fig = px.box(
    data,
    x='total_price',
    title='Order Value Boxplot',
    color_discrete_sequence = ["rebeccapurple"],
    width = 800,
    height = 500
)
fig.update_layout(
    title = dict(x = 0.5),
    title_font = dict(color = "green"),
    xaxis_title = "Prices",
    xaxis_title_font = dict(color = "red")
)
fig.show()

Order values are heavily concentrated in the lower price range, while a small subset of extremely high-value orders contributes disproportionately to overall revenue. The distribution exhibits extreme positive skewness and kurtosis, indicating substantial outlier presence and highly uneven customer spending behavior.

### Logistics

#### Delivery Days 

In [ ]:
stats(data , "delivery_days")

In [ ]:
data["delivery_days"].describe()

In [ ]:
plt.figure(figsize= (6,4))
sns.distplot(data["delivery_days"] , color = "orange")
plt.title("Delivery days distribtuion", color = "brown")
plt.xlabel("Delivery days")
plt.show()

In [ ]:
fig = px.box(data , x = "delivery_days", title = "Delivery days distribution", color_discrete_sequence=['green'])
fig.show()

Delivery performance is generally efficient, with most orders delivered within 7–10 days. However, the distribution is heavily right-skewed with substantial variability and extreme outliers, indicating that while most deliveries are timely, a subset of orders experiences significant delays. Missing delivery values are operationally meaningful and likely correspond to cancelled or undelivered orders.

#### Delivery Delays

In [ ]:
plt.figure(figsize= (6,4))
sns.distplot(data["delivery_delay"] , color = "orange")
plt.title("Delivery delays distribtuion", color = "brown")
plt.xlabel("Delivery delays")
plt.show()

In [ ]:
delayed_orders = data[data["delivery_delay"]>0]
delayed_orders["delivery_delay"].describe()

In [ ]:
stats(delayed_orders , "delivery_delay")

In [ ]:
plt.figure(figsize= (6,4))
sns.distplot(delayed_orders["delivery_delay"] , color = "green")
plt.title("Delayed days of Deliveries Distribtuion", color = "brown")
plt.xlabel(" days delayed")
plt.show()

In [ ]:
fig = px.box(data , x = "delivery_days", title = "Delivery days distribution", color_discrete_sequence=['teal'],
            height = 500, width = 700)
fig.update_layout (
    title_font = dict(color = "blue"), 
    title = dict(x = 0.5),
    xaxis_title = "No. of days delayed",
    font = dict(color = "black")
)
fig.show()

Delayed orders exhibit substantial variability and extreme right skewness. While the most common delay duration is only 1 day, a subset of orders experiences very large delays, significantly increasing the average delay time. The high kurtosis indicates the presence of severe outlier delays, suggesting operational inefficiencies in specific delivery cases.

### Satisfaction

In [ ]:
print("Total null values are : ",data["review_score"].isnull().sum().item())
print("Percent of Null values are : ",round(data["review_score"].isnull().mean(),2).item())

In [ ]:
data["review_score"].describe()

In [ ]:
query = """
    SELECT review_score , COUNT(*) AS Count
    FROM final_data
    WHERE review_score IS NOT NULL
    GROUP BY review_score
    ORDER BY Count DESC
    
"""

reviews_count = pd.read_sql(query, engine)
reviews_count

In [ ]:
fig = px.bar(
    reviews_count,
    x='review_score',
    y='Count',
    title='Review Score Distribution',
    height = 450,
    width = 900,
    color_discrete_sequence = ["seagreen"],
    text = "Count"
)

fig.update_layout(
    title = dict(x = 0.5),
    title_font = dict(color = "red"),
    xaxis_title = "Review Score"
)

fig.show()

Review ratings are strongly concentrated toward higher values, indicating generally positive customer experiences. However, the presence of low ratings and moderate variability suggests that certain operational issues still negatively impact a subset of customers.

### Payment 

In [ ]:
data["payment_type"].describe()

In [ ]:
query = """
            SELECT payment_type , COUNT(order_id) AS count
            FROM final_data
            WHERE payment_type IS NOT NULL
            GROUP BY payment_type
        """
payment_type_count = pd.read_sql(query,engine)
payment_type_count

In [ ]:
plt.figure(figsize = (8,4))
sns.barplot(x = payment_type_count["payment_type"] , y= payment_type_count["count"] )
plt.title(" Payment type Distribution", color = "green")
plt.xticks(rotation = 45, color = "red")
plt.xlabel("Payment Type", color = "orange")
plt.ylabel("Count", color = "orange")
plt.show()

In [ ]:
fig = px.pie(names = payment_type_count["payment_type"], values = payment_type_count["count"],
             title = "Payment type Distribution",
            height = 500,
            width = 500)
fig.update_traces(rotation = 30, direction  = "counterclockwise")
fig.show()

Credit cards overwhelmingly dominate customer payment behavior, suggesting strong adoption of digital and installment-based purchasing. Boleto also represents a meaningful share of transactions, emphasizing the continued relevance of alternative payment systems in Brazil. Debit card usage remains comparatively low, indicating customer preference toward more flexible payment options.

## 2. Bivariate Analysis

In [ ]:
data

####  Monthly Revenue Analysis

In [ ]:
query = """
            SELECT MONTH(order_purchase_timestamp) as Month_num, order_month , 
            ROUND(SUM(total_price) ,2) AS Revenue, COUNT(DISTINCT order_id) AS Total_Order
            FROM final_data
            GROUP BY order_month
            ORDER BY Month_num
         """
monthly_revenue = pd.read_sql(query, engine)
print("Monthly Revenue is : ")
monthly_revenue

In [ ]:
plt.figure(figsize = (8,4))
sns.lineplot(x = monthly_revenue["order_month"] , y= monthly_revenue["Revenue"] )
plt.title("Monthly Revenue Trend Analysis", color = "green")
plt.xticks(rotation = 45, color = "red")
plt.xlabel("Month")
plt.ylabel("Revenue (in millions)")
plt.show()

Revenue peaks during mid-year months, potentially reflecting seasonal purchasing behavior or increased marketplace engagement during that period.

#### Delivery Delay vs Review Score


In [ ]:
query = """
    SELECT review_score , AVG(delivery_delay) AS avg_delay, COUNT(*) AS Total_orders 
    FROM final_data
    WHERE review_score IS NOT NULL AND delivery_delay>0
    GROUP BY review_score
    ORDER BY review_score;
"""

delivery_delay_impact = pd.read_sql(query, engine)
delivery_delay_impact

In [ ]:
fig = px.line(delivery_delay_impact , x = "review_score" , y = "avg_delay", markers = True,
              title = "Impact of Delayed Deliveries On Reviews", 
              height = 400 , width = 800,
              color_discrete_sequence = ["darkcyan"])
fig.update_layout(
    title = dict(x = 0.5),
    title_font = dict(color = "black"),
    xaxis_title = "Review Score",
    yaxis_title = "Average Delay",
    xaxis_title_font = dict(color = "orange"),
    yaxis_title_font = dict(color = "orange")
)

Insight : Customers who provide lower review scores experience substantially longer delivery delays compared to highly satisfied customers. This suggests that delivery performance is a major driver of customer satisfaction and plays a critical role in shaping overall review behavior.

Recommendation : Reducing delivery delays could significantly improve customer satisfaction and platform ratings. Operational improvements in logistics and shipment tracking may help reduce negative customer experiences.

#### Late Vs On-time/Early Delivery

In [ ]:
query = """
            WITH delivery_count AS (
                                    SELECT 
                                        CASE 
                                            WHEN delivery_delay >0 THEN "Late"
                                            ELSE "On-time/Early"
                                        END AS delivery_status,
                                        COUNT(*) AS total_orders
                                    FROM final_data
                                    GROUP BY delivery_status
                                    )
            SELECT delivery_status, total_orders , 
                ROUND(total_orders*100.0/SUM(total_orders) OVER(),2) AS Percentage
            FROM delivery_count
"""
delivery_status = pd.read_sql(query, engine)
delivery_status

In [ ]:
fig = px.pie(delivery_status , values= "total_orders", names = "delivery_status",
             hole = 0.5, title = "On-time/Early Vs Late Deliveries",
            height = 480 , width = 600)
fig.update_layout(
    title = {"x":0.5}
)
fig.show()

Insight : The majority of orders are delivered on or before the estimated delivery date, indicating generally efficient logistics operations. However, delayed deliveries, although relatively infrequent, represent a critical operational issue because they are strongly associated with lower customer review scores and reduced customer satisfaction.

Recommendation : Improving delivery reliability and minimizing delayed shipments could significantly enhance customer satisfaction and overall platform experience.

#### Top States by Revenue

In [ ]:
query = """
            SELECT 
                customer_state, SUM(total_price) AS Revenue,
                COUNT(DISTINCT order_id) AS Total_Orders,
                ROUND(AVG(total_price), 2) AS Avg_Order_Value
            FROM final_data
            GROUP BY customer_state
            ORDER BY Revenue DESC
            LIMIT 10
        """
top_performing_states = pd.read_sql(query, engine)
top_performing_states

In [ ]:
fig = px.bar(
    top_performing_states,
    x='Revenue',
    y='customer_state',
    orientation='h',
    text='Revenue',
    title='Top 10 States by Revenue',
    color='Revenue'
)

fig.update_layout(
    title={'x':0.5},
    yaxis_title='State',
    xaxis_title='Revenue'
)

fig.show()

Revenue generation is heavily concentrated in SP, which significantly outperforms all other states in both order volume and total revenue. However, some smaller states exhibit comparatively higher average order values, indicating regional differences in customer purchasing behavior and spending patterns.

#### Payment Type vs Revenue

In [ ]:
query = """
            SELECT month(order_purchase_timestamp) as Month_num , order_month AS Month , ROUND(SUM(total_price),2) AS Revenue
            FROM final_data
            WHERE total_price IS NOT NULL 
            GROUP BY Month
            ORDER BY Month_num
        """
monthly_revenue  = pd.read_sql(query , engine)
monthly_revenue["Revenue_growth"] = round(monthly_revenue["Revenue"].pct_change(),2)

In [ ]:
monthly_revenue

In [ ]:
fig = px.line(monthly_revenue , x = "Month" , y = "Revenue_growth", 
             title = " Monthly Revenue Growth percentage",
             height = 500 , width = 900,
             color_discrete_sequence = ["mediumspringgreen"],
             text = "Revenue_growth",)

fig.update_layout(
    title = {"x":0.5},
    title_font = {"color":"brown"},
    xaxis_title = "Month",
    yaxis_title = "Growth",
    yaxis_title_font = {"color":"brown"},
    xaxis_title_font = {"color":"brown"}
    
)
fig.show()

Monthly revenue growth demonstrates considerable variability across the year, indicating fluctuating customer purchasing behavior and possible seasonal demand patterns. While several months show strong positive growth, significant declines are also observed, particularly in September. The sharp recovery during October and November suggests resilient market demand and strong late-year transaction activity.

#### Delivery days vs Total price


In [ ]:
query = """
            SELECT 
                CASE 
                    WHEN total_price < 100 THEN 'Low Value'
                    WHEN total_price BETWEEN 100 AND 500 THEN 'Medium Value'
                    ELSE 'High Value'
                END AS price_category,
            
                ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
            
                COUNT(*) AS total_orders
            
            FROM final_data
            
            WHERE delivery_days IS NOT NULL
            
            GROUP BY price_category;
"""

In [ ]:
price_delivery = pd.read_sql(query, engine)
price_delivery

In [ ]:
fig = px.bar( price_delivery, x='price_category',y='avg_delivery_days',
             text='avg_delivery_days',
             color='price_category',      
             title='Average Delivery Time by Order Value',
             height = 400,
             width = 850
        )

fig.update_layout(
    title={'x':0.5},
    xaxis_title='Order Value Category',
    yaxis_title='Average Delivery Days'
)

fig.show()

Higher-value orders tend to require longer delivery times, potentially due to increased logistics complexity and specialized handling requirements.

In [ ]:
filtered_df = data[
    (data['delivery_days'] < 100) &
    (data['total_price'] < 3000)
]

In [ ]:
fig = px.scatter(
    filtered_df,

    x='total_price',
    y='delivery_days',

    opacity=0.5,

    title='Delivery Days vs Total Price'
)

fig.show()

In [ ]:
filtered_df[['total_price', 'delivery_days']].corr()

Delivery time shows a very weak positive relationship with order value. Although higher-value orders exhibit slightly longer average delivery durations in grouped analysis, the overall correlation between total price and delivery days is minimal, indicating that order value is not a strong predictor of delivery performance.

#### Freight Cost Analysis

In [ ]:
query = """
    SELECT 
    ROUND(AVG(freight_value), 2) AS avg_freight,
    ROUND(AVG(price), 2) AS avg_product_price
    FROM final_data;
"""
freight_price = pd.read_sql(query, engine)
freight_price

In [ ]:
fig = px.scatter(data,x='price',y='freight_value',
    opacity=0.4,
    title='Freight Cost vs Product Price',
    color_discrete_sequence = ["maroon"],
    width = 800
)
fig.update_layout(
    title = {"x":0.5},
    title_font = {"color" : "green"},
    xaxis_title = "Price of product",
    yaxis_title = "Shipping cost",
    xaxis_title_font = {"color" : "blue"},
    yaxis_title_font = {"color" : "blue"},
)

fig.show()

In [ ]:
data[["total_price",'shipping_ratio']].corr()

Shipping ratio exhibits a moderate negative relationship with product price, indicating that lower-priced products tend to carry disproportionately higher shipping costs relative to their value, while shipping represents a much smaller proportion of cost for high-value products.

Optimizing shipping costs for lower-priced products could improve customer purchasing attractiveness and reduce perceived pricing burden.

#### State vs Revenue vs Avg Delivery Days

In [ ]:
query = """
            SELECT 
                customer_state,
                ROUND(SUM(total_price), 2) AS Revenue,
                ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
                COUNT(DISTINCT order_id) AS total_orders
            FROM final_data
            GROUP BY customer_state
            ORDER BY Revenue DESC
            LIMIT 10;
"""
result = pd.read_sql(query , engine)
result

The analysis reveals significant regional differences in marketplace performance. SP dominates both revenue generation and order volume while simultaneously maintaining the fastest average delivery times, suggesting highly efficient logistics operations in the platform’s strongest market. In contrast, several lower-revenue states experience substantially longer delivery durations, highlighting possible regional fulfillment inefficiencies and uneven operational performance across markets.

#### Repeat Customer Analysis

In [ ]:
query = """
        WITH total_orders AS(
                            SELECT 
                                customer_unique_id ,
                                COUNT(DISTINCT(order_id)) AS orders
                                FROM final_data
                                GROUP BY customer_unique_id
                                    
        )
        SELECT 
            CASE 
                WHEN orders = 1 
                THEN "One time customer"
                ELSE "Repeat customer"
            END AS customer_type,
            COUNT(*) AS total_customers
        FROM total_orders
        GROUP BY customer_type
                
        """
repeat_customers = pd.read_sql(query , engine)
repeat_customers

In [ ]:
fig = px.pie(
    repeat_customers,

    names='customer_type',
    values='total_customers',

    hole=0.5,

    title='Repeat vs One-Time Customers',
    width = 500,
    height = 600
)

fig.update_layout(
    title={'x':0.5},
    title_font = {"color" : "red" }
)

fig.show()

The platform appears to rely heavily on continuous acquisition of new customers rather than sustained repeat purchasing behavior, which may increase long-term customer acquisition costs.

The business could improve retention through:

1. loyalty programs
2. personalized recommendations
3. targeted email campaigns
4. improved post-purchase engagement
5. better delivery/customer experience

#### Customer Lifetime Value (CLV)

In [ ]:
query = """
WITH order_value AS (
    SELECT 
        order_id,
        customer_unique_id,
        MAX(total_price) AS order_total
    FROM final_data
    WHERE total_price IS NOT NULL
    GROUP BY order_id, customer_unique_id
)

SELECT 
    customer_unique_id,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(order_total), 2) AS customer_lifetime_value,
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM order_value
GROUP BY customer_unique_id
ORDER BY customer_lifetime_value DESC;
"""

In [ ]:
clv = pd.read_sql(query, engine)
clv

In [ ]:
filtered_clv = clv[
    clv['customer_lifetime_value'] < 1000
]

In [ ]:
fig = px.histogram(
    clv,

    x='customer_lifetime_value',

    nbins=50,

    title='Distribution of Customer Lifetime Value'
)

fig.update_layout(
    title={'x':0.5},
    xaxis_title='Customer Lifetime Value',
    yaxis_title='Customer Count'
)

fig.show()

Customer spending distribution is highly uneven, with a small number of customers contributing disproportionately high purchase values relative to the broader customer base.